# Evaluating Performance of Linear Chain CRF on the CoNLL-2003 Dataset

## Load Dependencies

In [1]:
from collections import Counter
from datasets import concatenate_datasets, DatasetDict, load_dataset
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF

## Set Global Parameters

In [2]:
BATCH_N = 32
EMBED_DIM = 300
LEARNING_RATE = 1e-4
NUM_EPOCHS = 60
AFFIX_LEN = 4
AFFIX_VOCAV_SIZE = 500

## Load Data

In [3]:
# Data downloaded from https://nlp.stanford.edu/projects/glove/
glove_path = "data/wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"

embeddings_dict = {}

with open(file = glove_path, mode = 'r', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = ''.join(values[:len(values)-300])
        vector = np.asarray(values[len(values)-300:], "float32")
        embeddings_dict[word] = vector

In [4]:
BASE = "https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003"

dataset = DatasetDict({
    split: load_dataset("parquet", data_files={split: f"{BASE}/{split}/0000.parquet"}, split=split)
    for split in ["train", "validation", "test"]
})

NUM_POS_CLASSES = dataset['train'].features['pos_tags'].feature.num_classes
NUM_CHUNK_CLASSES = dataset['train'].features['chunk_tags'].feature.num_classes

## Load Gazetteer

In [5]:
import zipfile
import gzip

gazetteers = {
    "PER": set(), "LOC": set(), "ORG": set(), "MISC": set()
}

per_files = [
    'gazetteers/People.gz',
    'gazetteers/People.Famous.gz',
    'gazetteers/People.Gender.Female.gz',
    'gazetteers/People.Gender.Male.gz',
    'gazetteers/Person.gz',
    'gazetteers/People.Politicians.gz',
]
loc_files = [
    'gazetteers/Locations.gz',
    'gazetteers/Locations.Cities.gz',
    'gazetteers/Locations.Countries.gz',
    'gazetteers/Locations.States.gz',
    'gazetteers/Locations.Regions.gz',
    'gazetteers/Locations.Generic.gz',
    'gazetteers/Locations.Cities.US.gz',
    'gazetteers/Locations.Cities.Canada.gz',
    'gazetteers/Locations.Cities.Europe.gz',
]
org_files = [
    'gazetteers/Organizations.gz',
    'gazetteers/Corporations.gz',
    'gazetteers/Government.gz',
    'gazetteers/PoliticalParties.gz',
    'gazetteers/Organizations.EducationalInstitutions.gz',
]
misc_files = [
    'gazetteers/Nationalities.gz',
    'gazetteers/Languages.gz',
    'gazetteers/EthnicGroups.gz',
]

mapping = {
    "PER": per_files,
    "LOC": loc_files,
    "ORG": org_files,
    "MISC": misc_files
}

with zipfile.ZipFile("data/gazetteers-1.5.jar", "r") as jar: # download at https://cogcomp.seas.upenn.edu/m2repo/edu/illinois/cs/cogcomp/resources/gazetteers/1.5/gazetteers-1.5.jar
    for ent, files in mapping.items():
        for file in files:
            with jar.open(file) as f:
                with gzip.open(f, 'rt', encoding='utf-8', errors='ignore') as gz:
                    for i, line in enumerate(gz):
                        token = line.strip().lower()
                        if "list of" not in token and len(token) > 1:
                            if i % 100_000 == 0:
                                print(f"Item {i}: {token}")
                            gazetteers[ent].add(token)


Item 0: "big nick" nicholas
Item 100000: bob vincent
Item 200000: donald ben marsh
Item 300000: glover morrill allen
Item 400000: jessica dubroff
Item 500000: leon c. collins
Item 600000: namaa alward
Item 700000: robert magaw
Item 800000: timur kulibayev
Item 0: aaadietya pandey
Item 100000: del shofner
Item 200000: james hartness
Item 300000: masaaki endoh
Item 400000: shawn cronin
Item 0: aṣa
Item 100000: saloni aswani
Item 0: a. a. aboutaleb
Item 100000: christian mclaughlin
Item 200000: george k. mackenzie
Item 300000: john hester
Item 400000: miguel angel garcia
Item 500000: rudolf spillmann
Item 600000: ye xuanping
Item 0: !all-time quarterback!
Item 100000: ali akbar
Item 200000: atake tynay biy uulu
Item 300000: bryton
Item 400000: clem de rosa
Item 500000: diocese of nassau and the bahamas and the turks and caicos islands
Item 600000: enzo salvi
Item 700000: fritz amling
Item 800000: gustaf trolle
Item 900000: i gusti ketut jelantik
Item 1000000: jay trumbull
Item 1100000: jo

In [6]:
# print some examples from the gazetteer
for ent, val in gazetteers.items():
    for i, v in enumerate(val):
        if i+1 > 2: # only print 2 examples per entity
            break
        else:
            print(f"{ent}: {v}")

PER: yolanda ivonne móntez farrington
PER: sir panaganti ramarayaningar
LOC: gull lake no. 139, saskatchewan
LOC: wakasa, fukui
ORG: skene records
ORG: syrian arab airline
MISC: oneida
MISC: bermudians


## Define Model and Utilities

In [7]:
def get_affixes(corpus, kind, affix_length):

    affs = Counter()
    for seq in corpus:
        eligible_toks = [t.lower() for t in seq if len(t) > 2]

        for t in eligible_toks:
            index = min(affix_length, len(t)-1)
            for i in range(index):
                if kind == "prefix":
                    val = t[:(index-i)]
                elif kind == "suffix":
                    val = t[-1*(index-i):]
                else:
                    raise ValueError("Only prefix and suffix are supported.")
                if len(val) > 1:
                    affs[val] += 1
    return affs

def get_top_affixes(corpus, top=AFFIX_VOCAV_SIZE, affix_length=AFFIX_LEN):
    all_pre = get_affixes(corpus, kind="prefix", affix_length=affix_length)
    top_pre = all_pre.most_common(top)
    all_suf = get_affixes(corpus, kind="suffix", affix_length=affix_length)
    top_suf = all_suf.most_common(top)

    return top_pre, top_suf

In [8]:
sent_1 = ["We", "synergize", "at", "work"]
sent_2 = ["I", "require", "the", "next", "size"]
sent_3 = ["I", "will", "synthesize", "it"]
sent_4 = ["I", "love", "my", "synth"]

corpus = [sent_1, sent_2, sent_3, sent_4]

pres, sufs = get_top_affixes(corpus, top=5)

print("Prefix:", pres)
print("Suffix:", sufs)

sent_test = ["We", "realize", "it", "is", "synthetic"]

print("\n---Prefix Test---")
for t in sent_test:
    for p in pres:
        if t.startswith(p[0]):
            print(p[0])
print("\n---Suffix Test---")
for t in sent_test:
    for s in sufs:
        if t.endswith(s[0]):
            print(s[0])

Prefix: [('syn', 3), ('sy', 3), ('synt', 2), ('syne', 1), ('wor', 1)]
Suffix: [('ize', 3), ('ze', 3), ('gize', 1), ('ork', 1), ('rk', 1)]

---Prefix Test---
syn
sy
synt

---Suffix Test---
ize
ze


In [9]:
OOV_VECTOR = np.zeros(EMBED_DIM, dtype="float32")

class NERDataset(Dataset):
    '''
    Prepares dataset for Pytorch's `DataLoader`.
    '''
    def __init__(
            self, 
            tokens, 
            tags, 
            pos_tags, 
            c_tags, 
            embed_map, 
            affix_len=AFFIX_LEN, 
            affix_size=AFFIX_VOCAV_SIZE,
            prefix_vocab=None,
            suffix_vocab=None,
            gazetteers = None
        ):
        self.samples = [(t, g, p, c) for t, g, p, c in zip(tokens, tags, pos_tags, c_tags) if len(t)>0]
        self.embed_map = embed_map
        self.affix_len = affix_len
        if prefix_vocab is None:
            top_prefix, top_suffix = get_top_affixes(tokens, top=affix_size, affix_length=affix_len)
            self.prefix_vocab = {aff:i+1 for i, (aff, _) in enumerate(top_prefix)}
        else:
            self.prefix_vocab = prefix_vocab
        if suffix_vocab is None:
            top_prefix, top_suffix = get_top_affixes(tokens, top=affix_size, affix_length=affix_len)
            self.suffix_vocab = {aff:i+1 for i, (aff, _) in enumerate(top_suffix)}
        else:
            self.suffix_vocab = suffix_vocab
        self.affix_size = affix_size + 1
        self.gazetteers = gazetteers

    def get_multihot(self, tok, kind):
        vec = torch.zeros(self.affix_size)
        index = min(self.affix_len, len(tok)-1)
        for i in range(index):
            if kind == "prefix":
                aff = tok.lower()[:(index-i)]
                if aff in self.prefix_vocab:
                    vec[self.prefix_vocab[aff]] = 1.0
            elif kind == "suffix":
                aff = tok.lower()[-1*(index-i):]
                if aff in self.suffix_vocab:
                    vec[self.suffix_vocab[aff]] = 1.0
            else:
                raise ValueError("Only prefix and suffix are supported.")
        return vec
    
    def get_bin(self, t):
        if len(t) < 3:
            return torch.tensor([1, 0, 0], dtype=torch.float32)
        if len(t) < 9:
            return torch.tensor([0, 1, 0], dtype=torch.float32)
        else:
            return torch.tensor([0, 0, 1], dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        toks, tags, pos, chnk = self.samples[idx]
        vecs = torch.stack([
            torch.cat([
                torch.from_numpy(self.embed_map.get(t, self.embed_map.get(t.lower(), OOV_VECTOR))),
                torch.nn.functional.one_hot(torch.tensor(p), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(pos[max(i-1, 0)]), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(pos[min(i+1, len(toks)-1)]), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(c), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(chnk[max(i-1, 0)]), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(chnk[min(i+1, len(toks)-1)]), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(int(t.isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in t]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in toks[max(i-1, 0)]]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in toks[min(i+1, len(toks)-1)]]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.islower())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].islower())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].islower())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.isdigit() and len(t) == 4)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].isdigit() and len(toks[max(i-1, 0)]) == 4)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].isdigit() and len(toks[min(i+1, len(toks)-1)]) == 4)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t[0].isupper() and i > 0)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)][0].isupper() and i > 0)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)][0].isupper() and i > 0)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.istitle())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in t)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in toks[max(i-1, 0)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in toks[min(i+1, len(toks)-1)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('.' in t)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('.' in toks[max(i-1, 0)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('.' in toks[min(i+1, len(toks)-1)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(i==0)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(i==len(toks)-1)), num_classes=2),
                self.get_bin(t),
                self.get_bin(toks[max(i-1, 0)]),
                self.get_bin(toks[min(i+1, len(toks)-1)]),
                self.get_multihot(t, kind="suffix"),
                self.get_multihot(t, kind="prefix"),
                torch.tensor([
                    float(t.lower() in self.gazetteers['PER'] if self.gazetteers else 0),
                    float(t.lower() in self.gazetteers['LOC'] if self.gazetteers else 0),
                    float(t.lower() in self.gazetteers['ORG'] if self.gazetteers else 0),
                    float(t.lower() in self.gazetteers['MISC'] if self.gazetteers else 0)
                ], dtype=torch.float32),
                torch.tensor([
                    float(toks[max(i-1, 0)].lower() in self.gazetteers['PER'] if self.gazetteers else 0),
                    float(toks[max(i-1, 0)].lower() in self.gazetteers['LOC'] if self.gazetteers else 0),
                    float(toks[max(i-1, 0)].lower() in self.gazetteers['ORG'] if self.gazetteers else 0),
                    float(toks[max(i-1, 0)].lower() in self.gazetteers['MISC'] if self.gazetteers else 0)
                ], dtype=torch.float32),
                torch.tensor([
                    float(toks[min(i+1, len(toks)-1)].lower() in self.gazetteers['PER'] if self.gazetteers else 0),
                    float(toks[min(i+1, len(toks)-1)].lower() in self.gazetteers['LOC'] if self.gazetteers else 0),
                    float(toks[min(i+1, len(toks)-1)].lower() in self.gazetteers['ORG'] if self.gazetteers else 0),
                    float(toks[min(i+1, len(toks)-1)].lower() in self.gazetteers['MISC'] if self.gazetteers else 0)
                ], dtype=torch.float32)
            ])
            for i, (t, p, c) in enumerate(zip(toks, pos, chnk))
        ])
        return vecs, torch.tensor(tags, dtype=torch.long), toks


def collate_fn(batch):
    '''
    Defines padding and masking for Pytorch's `DataLoader`.
    Returns a tuple.
    '''
    seqs, tags, toks = zip(*batch)
    X = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    y = pad_sequence(tags, batch_first=True, padding_value=-1)
    mask = (y != -1).bool()
    return X, y, mask, toks

In [10]:
class LinearChainCRF(nn.Module):
    ''' 
    A simple linear chain CRF for NLP tasks.
    First flattens embeddings to match number of NER tags,
    then passes results to the CRF.
    '''
    def __init__(self, embed_dim, num_tags):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_tags)
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, x, tags, mask):
        emissions = self.linear(x)
        return -self.crf(emissions, tags, mask=mask)

    def decode(self, x, mask):
        emissions = self.linear(x)
        return self.crf.decode(emissions, mask=mask)

## Training

In [ ]:
num_tags = dataset['train'].features['ner_tags'].feature.num_classes
model = LinearChainCRF(EMBED_DIM+3*NUM_POS_CLASSES+3*NUM_CHUNK_CLASSES+30*2+2*(AFFIX_VOCAV_SIZE+1)+3*4+3*3, num_tags)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(
    NERDataset(
        dataset["train"]["tokens"], 
        dataset["train"]["ner_tags"], 
        dataset['train']['pos_tags'], 
        dataset['train']['chunk_tags'], 
        embeddings_dict,
        gazetteers=gazetteers
    ),
    batch_size=BATCH_N, shuffle=True, collate_fn=collate_fn
)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for X, y, mask, _ in train_loader:
        optimizer.zero_grad()
        loss = model(X, y, mask) / mask.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss:.4f}")


## Validation

In [ ]:
ner_labels = dataset['train'].features['ner_tags'].feature.names

train_instance = NERDataset(
        dataset['train']["tokens"], 
        dataset['train']["ner_tags"], 
        dataset['train']['pos_tags'], 
        dataset['train']['chunk_tags'], 
        embeddings_dict,
        gazetteers=gazetteers
    )

val_loader = DataLoader(
    NERDataset(
        dataset["validation"]["tokens"], 
        dataset["validation"]["ner_tags"], 
        dataset['validation']['pos_tags'], 
        dataset['validation']['chunk_tags'], 
        embeddings_dict,
        prefix_vocab=train_instance.prefix_vocab,
        suffix_vocab=train_instance.suffix_vocab,
        gazetteers=train_instance.gazetteers
    ),
    batch_size=BATCH_N, shuffle=False, collate_fn=collate_fn
)

model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for X, y, mask, _ in val_loader:
        batch_preds = model.decode(X, mask)
        for pred_seq, gold_seq, m in zip(batch_preds, y, mask):
            valid_len = m.sum().item()
            all_preds.append([ner_labels[i] for i in pred_seq[:valid_len]])
            all_golds.append([ner_labels[i] for i in gold_seq[:valid_len].tolist()])
            

In [ ]:
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

print(f"F1-score: {f1_score(all_golds, all_preds):.2f}")
print(classification_report(all_golds, all_preds))

## Results on Test Set

In [11]:
train_dataset = concatenate_datasets([dataset['train'], dataset['validation']])
num_tags = train_dataset.features['ner_tags'].feature.num_classes
model = LinearChainCRF(EMBED_DIM+3*NUM_POS_CLASSES+3*NUM_CHUNK_CLASSES+30*2+2*(AFFIX_VOCAV_SIZE+1)+3*4+3*3, num_tags)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(
    NERDataset(
        train_dataset["tokens"], 
        train_dataset["ner_tags"], 
        train_dataset['pos_tags'], 
        train_dataset['chunk_tags'], 
        embeddings_dict,
        gazetteers = gazetteers
    ),
    batch_size=BATCH_N, shuffle=True, collate_fn=collate_fn
)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for X, y, mask, _ in train_loader:
        optimizer.zero_grad()
        loss = model(X, y, mask) / mask.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss:.4f}")

Epoch 1  loss=588.9703
Epoch 2  loss=262.1859
Epoch 3  loss=191.6037
Epoch 4  loss=153.8184
Epoch 5  loss=129.5698
Epoch 6  loss=112.5539
Epoch 7  loss=99.6851
Epoch 8  loss=89.6337
Epoch 9  loss=81.7371
Epoch 10  loss=75.3908
Epoch 11  loss=69.8900
Epoch 12  loss=65.4153
Epoch 13  loss=61.4708
Epoch 14  loss=57.9685
Epoch 15  loss=55.2740
Epoch 16  loss=52.8632
Epoch 17  loss=50.5345
Epoch 18  loss=48.4706
Epoch 19  loss=46.6973
Epoch 20  loss=45.1246
Epoch 21  loss=43.8528
Epoch 22  loss=42.4772
Epoch 23  loss=41.0811
Epoch 24  loss=40.1531
Epoch 25  loss=39.0670
Epoch 26  loss=38.0293
Epoch 27  loss=37.4497
Epoch 28  loss=36.2409
Epoch 29  loss=35.5822
Epoch 30  loss=34.7474
Epoch 31  loss=34.0320
Epoch 32  loss=33.5158
Epoch 33  loss=33.0683
Epoch 34  loss=32.2550
Epoch 35  loss=31.6609
Epoch 36  loss=31.1276
Epoch 37  loss=30.5596
Epoch 38  loss=30.1282
Epoch 39  loss=29.6492
Epoch 40  loss=29.1563
Epoch 41  loss=28.8194
Epoch 42  loss=28.4837
Epoch 43  loss=28.0455
Epoch 44  loss

In [12]:
ner_labels = train_dataset.features['ner_tags'].feature.names

train_instance = NERDataset(
        train_dataset["tokens"], 
        train_dataset["ner_tags"], 
        train_dataset['pos_tags'], 
        train_dataset['chunk_tags'], 
        embeddings_dict,
        gazetteers=gazetteers
    )

test_loader = DataLoader(
    NERDataset(
        dataset["test"]["tokens"], 
        dataset["test"]["ner_tags"], 
        dataset['test']['pos_tags'], 
        dataset['test']['chunk_tags'], 
        embeddings_dict,
        prefix_vocab=train_instance.prefix_vocab,
        suffix_vocab=train_instance.suffix_vocab,
        gazetteers=train_instance.gazetteers

    ),
    batch_size=BATCH_N, shuffle=False, collate_fn=collate_fn
)

model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for X, y, mask, toks in test_loader:
        batch_preds = model.decode(X, mask)
        for pred_seq, gold_seq, m, t in zip(batch_preds, y, mask, toks):
            valid_len = m.sum().item()
            all_preds.append([ner_labels[i] for i in pred_seq[:valid_len]])
            all_golds.append([ner_labels[i] for i in gold_seq[:valid_len].tolist()])


In [14]:
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

print(f"F1-score: {f1_score(all_golds, all_preds):.2f}")
print(classification_report(all_golds, all_preds))

F1-score: 0.87
              precision    recall  f1-score   support

         LOC       0.87      0.91      0.89      1668
        MISC       0.78      0.77      0.77       702
         ORG       0.83      0.81      0.82      1661
         PER       0.94      0.92      0.93      1617

   micro avg       0.87      0.87      0.87      5648
   macro avg       0.85      0.85      0.85      5648
weighted avg       0.87      0.87      0.87      5648

